In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
#==========================================================
# Load Packages 
#==========================================================
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import sys
sys.path.append('/project/IZZY/MolRepres/Methods/')                  # Path where the training, evaluating function is located 
from sklearn.preprocessing import StandardScaler                     # Tool for normalizing data (zero mean, unit variance)
from torch_geometric.datasets import QM9                             # Dataset utilities for 3D molecular graphs (QM9 with 3D coordinates)
from torch_geometric.nn import GATConv, global_mean_pool             # Graph Neural Network layers and pooling operations from PyTorch Geometric
from torch_geometric.loader import DataLoader                        # DataLoader to batch and shuffle molecular graph data
from torch.optim import Adam                                         # Optimizer (Adam) and learning rate scheduler (StepLR)
from torch.optim.lr_scheduler import StepLR                          # Progress bar for training/evaluation loops
from tqdm import tqdm                                                # Unit conversion constants (Hartree, eV, Bohr, Angstrom) from ASE
from ase.units import Hartree, eV, Bohr, Ang                         # TensorBoard writer for tracking metrics and visualizing training progress
from torch.utils.tensorboard import SummaryWriter                    # TensorBoard writer for tracking metrics and visualizing training progress
from train_eval import train_epoch, evaluate                         # Importing the training and evaluating function

In [3]:
#==========================================================
# GPU Device
#==========================================================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # This line checks if a CUDA-enabled GPU is available. If yes, computations will be performed on the GPU for faster training. Otherwise, it falls back to using the CPU.
# Print which device is being used 
print("Using device:", device)

Using device: cuda


In [4]:
#==============================================================
# Load dataset and split
#==============================================================

# Load the QM9 dataset 
dataset = QM9(root='data/QM9')

# Count how many molecular graphs are in the dataset
num_mols = len(dataset)
print(f"Total QM9 molecules: {num_mols}")

# Define the desired split sizes for training, validation, and testing
train_size, valid_size = 110000, 10000
# Ensure the sum of train and validation sizes does not exceed dataset size
assert train_size + valid_size < num_mols, "Split sizes too large for dataset"

# Randomly shuffle all molecule indices
perm = torch.randperm(num_mols, generator=torch.Generator().manual_seed(42))

# Use the shuffled indices to create non-overlapping subsets
train_idx = perm[:train_size]
valid_idx = perm[train_size:train_size+valid_size]
test_idx  = perm[train_size+valid_size:]

# Build subsets based on the index splits
train_dataset = dataset[train_idx]
valid_dataset = dataset[valid_idx]
test_dataset  = dataset[test_idx]

Total QM9 molecules: 130831


In [5]:
#======================================================================
# Define the QM9 Targets
#======================================================================
PROPERTY_NAMES = [
    'μ (D)', 'α (Ang³)', 'ε_HOMO (eV)', 'ε_LUMO (eV)', 'Δε (eV)', '⟨R²⟩ (Ang²)',
    'ZPVE (eV)', 'U₀ (eV)', 'U (eV)', 'H (eV)', 'G (eV)', 'c_v', 'U₀_atom',
    'U_atom', 'H_atom', 'G_atom', 'A (GHz)', 'B (GHz)', 'C (GHz)'
] 
# ======================================================================
# QM9 Unit Conversion Factors
# ======================================================================
def get_qm9_conversions_tensor(device):
    return torch.tensor([
        1.0, Bohr**3 / Ang**3, Hartree / eV, Hartree / eV, Hartree / eV,
        Bohr**2 / Ang**2, Hartree / eV, Hartree / eV, Hartree / eV,
        Hartree / eV, Hartree / eV, 1.0, 1.0, Hartree / eV,
        Hartree / eV, Hartree / eV, 1.0, 1.0, 1.0
    ], dtype=torch.float, device=device)

In [6]:
#========================================================================
# Normalize all QM9 targets
#========================================================================
y_raw_all = dataset.data.y.clone().cpu()
conversions_cpu = get_qm9_conversions_tensor('cpu')
y_conv_all = y_raw_all * conversions_cpu.unsqueeze(0)

norm_stats = {'mean': [], 'std': []}
y_norm_all = torch.zeros_like(y_conv_all)

for i in range(y_conv_all.shape[1]):
    train_y_cpu = y_conv_all[train_idx.cpu(), i]
    mean_i = float(train_y_cpu.mean().item())
    std_i = float(train_y_cpu.std().item()) if train_y_cpu.std().item() != 0 else 1.0
    y_norm_all[:, i] = (y_conv_all[:, i] - mean_i) / std_i
    norm_stats['mean'].append(mean_i)
    norm_stats['std'].append(std_i)

dataset.data.y = y_norm_all.to(torch.float)
print("Normalization complete for all targets.")

/home/ismail/dig_envi/lib/python3.9/site-packages/torch_geometric/data/in_memory_dataset.py:300: UserWarning: It is not recommended to directly access the internal storage format `data` of an 'InMemoryDataset'. If you are absolutely certain what you are doing, access the internal storage via `InMemoryDataset._data` instead to suppress this warning. Alternatively, you can access stacked individual attributes of every graph via `dataset.{attr_name}`.
  warnings.warn(msg)


Normalization complete for all targets.


In [7]:
#===========================================================================
# Build GAT-Model 
#===========================================================================

# Number of input features per node (atom) in each molecular graph
in_channels = dataset.num_node_features
print("Node feature dim:", in_channels)

#--------------------------
# Define GAT-Model
#--------------------------
class GATModel(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, dropout=0.3, out_dim=19):
        """
        Simple 3-layer GCN for regression on molecular graphs.

        Args:
            in_dim(int): Dimension of node features
            hidden_dim(int): Number of hidden units in GCN layers 
            dropout (float): Dropout probability
        """
        
        super().__init__()
        # Three graph convolutional layers
        self.conv1 = GATConv(in_dim, hidden_dim)                  # Each layer updates node embeddings using neighbors' features
        self.conv2 = GATConv(hidden_dim, hidden_dim)
        self.conv3 = GATConv(hidden_dim, hidden_dim)

        # Fully connected layers to map pooled graph embedding => target value
        self.fc1 = nn.Linear(hidden_dim, 64)
        self.fc2 = nn.Linear(64, out_dim)                               # Output: single scalar target
        
        # Dropout for regularization
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, batch):
        """
        
        Forward pass of the GAT.

        Args:
            batch: a PyTorch Geometric batch containing:
                    - x: node features[num_nodes, in_dim]
                    - edge_index: edge list[2, num_edges]
                    - batch: batch vector mapping nodes to molecules

        Returns :
            Predicted target values for each molecule [batch_size, 1]
            
        """
        x, edge_index, batch_vec = batch.x, batch.edge_index, batch.batch

        # Apply three GAT layers with ReLU activation
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        x = F.relu(self.conv3(x, edge_index))
        
        # Global mean Pooling
        x = global_mean_pool(x, batch_vec)   # The global mean pool ing takes the mean of all atom embeddings in each molecule´s graph "one feat vector per molecule"
        
        # Decoder : it transfomrs the pooled molecular embedding into the predicted target value 
        x = F.relu(self.fc1(x)) 
        x = self.dropout(x)
        return self.fc2(x)

Node feature dim: 11


In [8]:
#==============================================================================================
# Trainig Loop
#==============================================================================================

def main():
    #------------------------------
    # Hyperparameters
    #------------------------------
    epochs = 1000                            # number of training epochs
    batch_size = 128                        # batch size for training
    vt_batch_size = 256                     # batch size for validation
    lr = 1e-5                               # learning rate
    lr_decay_step = 50                      # steps after which LR is decayed
    lr_decay_factor = 0.5                   # factor to decay learning rate
    weight_decay = 1e-4                     # L2 regularization
    save_dir = 'checkpoints_GAT'            # directory to save model checkpoints 
    log_dir = 'logs_GAT'                    # TensorBoard logs directory

    #Create directories if they don't exist
    os.makedirs(save_dir, exist_ok=True)
    os.makedirs(log_dir, exist_ok=True)

    # TensorBoard write for visualization of metrics
    writer = SummaryWriter(log_dir=log_dir)

    #-----------------------------
    # Dataloaders 
    #-----------------------------

    # Shuffled batches for training; sequential batches for validation/testing
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(valid_dataset, batch_size=vt_batch_size, shuffle=False)
    test_loader  = DataLoader(test_dataset, batch_size=vt_batch_size, shuffle=False)

    #----------------------------------------
    # Model
    #----------------------------------------
    # GCN model
    model = GATModel(in_channels).to(device)  # GCN model uncommented to use it 

    #----------------------------------------
    # Optimizer and Scheduler
    #----------------------------------------
    optimizer = Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = StepLR(optimizer, step_size=lr_decay_step, gamma=lr_decay_factor)

    # Track best validation/test performance
    best_mean_val=float('inf')
    best_val = [float('inf')] * len(PROPERTY_NAMES)
    best_test = [float('inf')] * len(PROPERTY_NAMES)

    # Print total number of trainabel parameters and target property
    print("#params:", sum(p.numel() for p in model.parameters()))
    print("Training for all 19 targets") 
    
    #--------------------------------------------
    # Training Loop
    #--------------------------------------------
    for epoch in range(1, epochs + 1):
        print(f"\n=== Epoch {epoch} ===")

        #Train for one epoch
        train_loss = train_epoch(model, train_loader, optimizer, device)

        # Evaluate on validation and test sets (MAE in original units)
        val_mae = evaluate(model, val_loader, device, norm_stats['mean'], norm_stats['std'])
        test_mae = evaluate(model, test_loader, device, norm_stats['mean'], norm_stats['std'])

        # Print epoch metrics
        print(f"Train loss (MSE): {train_loss:.6f}")
        for i, prop in enumerate(PROPERTY_NAMES):
            print(f"  {prop:15s} | Val MAE: {val_mae[i]:.6f} | Test MAE: {test_mae[i]:.6f}")
            writer.add_scalar(f'val_mae/{prop}', val_mae[i], epoch)
            writer.add_scalar(f'test_mae/{prop}', test_mae[i], epoch)
        #---------------------------------------------
        # Save checkpoint if validation improves 
        #---------------------------------------------
        mean_val_mae = sum(val_mae) / len(val_mae)
        
        # If mean validation MAE improved → save one best model
        if mean_val_mae < best_mean_val:
            best_mean_val = mean_val_mae
            best_val = val_mae
            best_test = test_mae
        
            save_path = os.path.join(save_dir, 'best_model.pt')
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'best_val': best_val,
                'best_test': best_test,
                'best_mean_val': best_mean_val
            }, save_path)
        
            print(f" Saved best overall model (mean validation MAE improved to {best_mean_val:.4f})")

        # Update learning rate according to scheduler
        scheduler.step()
    # Close TensorBoard writer
    writer.close()
    print("\nFinished training.")
    print("Best validation and test MAEs per property:")
    for prop, v, t in zip(PROPERTY_NAMES, best_val, best_test):
        print(f"  {prop:15s} | Best validation MAE: {v:.6f} | Test MAE at best val: {t:.6f}")

    #print("Best validation MAE:", best_val)
    #print("Test MAE at best val:", best_test)

# Run the training loop
if __name__ == "__main__":
    main()

#params: 44819
Training for all 19 targets

=== Epoch 1 ===


Train loss (MSE): 0.995164
  μ (D)           | Val MAE: 1.148002 | Test MAE: 1.166652
  α (Ang³)        | Val MAE: 0.922446 | Test MAE: 0.919312
  ε_HOMO (eV)     | Val MAE: 11.994886 | Test MAE: 12.015081
  ε_LUMO (eV)     | Val MAE: 28.026775 | Test MAE: 28.461033
  Δε (eV)         | Val MAE: 29.361506 | Test MAE: 29.501177
  ⟨R²⟩ (Ang²)     | Val MAE: 56.864582 | Test MAE: 56.737690
  ZPVE (eV)       | Val MAE: 18.801622 | Test MAE: 18.765802
  U₀ (eV)         | Val MAE: 22954.460938 | Test MAE: 22572.312500
  U (eV)          | Val MAE: 22537.615234 | Test MAE: 22165.380859
  H (eV)          | Val MAE: 23064.496094 | Test MAE: 22683.312500
  G (eV)          | Val MAE: 22421.144531 | Test MAE: 22054.361328
  c_v             | Val MAE: 3.177958 | Test MAE: 3.149277
  U₀_atom         | Val MAE: 7.957665 | Test MAE: 7.927421
  U_atom          | Val MAE: 214.861176 | Test MAE: 213.987045
  H_atom          | Val MAE: 215.766357 | Test MAE: 214.935852
  G_atom          | Val MAE: 197.80555

Train loss (MSE): 0.915479
  μ (D)           | Val MAE: 1.124715 | Test MAE: 1.143417
  α (Ang³)        | Val MAE: 0.824885 | Test MAE: 0.820241
  ε_HOMO (eV)     | Val MAE: 11.842900 | Test MAE: 11.871594
  ε_LUMO (eV)     | Val MAE: 25.119368 | Test MAE: 25.465755
  Δε (eV)         | Val MAE: 28.028616 | Test MAE: 28.131882
  ⟨R²⟩ (Ang²)     | Val MAE: 54.593975 | Test MAE: 54.564842
  ZPVE (eV)       | Val MAE: 14.199320 | Test MAE: 14.064335
  U₀ (eV)         | Val MAE: 21043.240234 | Test MAE: 20664.691406
  U (eV)          | Val MAE: 19626.410156 | Test MAE: 19239.957031
  H (eV)          | Val MAE: 21871.822266 | Test MAE: 21487.855469
  G (eV)          | Val MAE: 19244.968750 | Test MAE: 18851.242188
  c_v             | Val MAE: 2.798621 | Test MAE: 2.760988
  U₀_atom         | Val MAE: 6.289677 | Test MAE: 6.227507
  U_atom          | Val MAE: 166.149185 | Test MAE: 164.249100
  H_atom          | Val MAE: 163.961456 | Test MAE: 161.936829
  G_atom          | Val MAE: 160.94838

Train loss (MSE): 0.733778
  μ (D)           | Val MAE: 1.047203 | Test MAE: 1.063775
  α (Ang³)        | Val MAE: 0.643115 | Test MAE: 0.634384
  ε_HOMO (eV)     | Val MAE: 11.683653 | Test MAE: 11.734318
  ε_LUMO (eV)     | Val MAE: 22.947798 | Test MAE: 23.087009
  Δε (eV)         | Val MAE: 25.307463 | Test MAE: 25.385727
  ⟨R²⟩ (Ang²)     | Val MAE: 53.770924 | Test MAE: 53.875767
  ZPVE (eV)       | Val MAE: 9.059645 | Test MAE: 8.787106
  U₀ (eV)         | Val MAE: 18076.414062 | Test MAE: 17676.958984
  U (eV)          | Val MAE: 17879.632812 | Test MAE: 17496.537109
  H (eV)          | Val MAE: 18531.070312 | Test MAE: 18122.888672
  G (eV)          | Val MAE: 17669.871094 | Test MAE: 17292.007812
  c_v             | Val MAE: 2.515076 | Test MAE: 2.470301
  U₀_atom         | Val MAE: 4.143604 | Test MAE: 4.017103
  U_atom          | Val MAE: 111.532623 | Test MAE: 107.851776
  H_atom          | Val MAE: 109.160484 | Test MAE: 105.384026
  G_atom          | Val MAE: 108.356796 

Train loss (MSE): 0.671067
  μ (D)           | Val MAE: 1.020324 | Test MAE: 1.036038
  α (Ang³)        | Val MAE: 0.598891 | Test MAE: 0.589192
  ε_HOMO (eV)     | Val MAE: 11.653468 | Test MAE: 11.703572
  ε_LUMO (eV)     | Val MAE: 22.551550 | Test MAE: 22.681314
  Δε (eV)         | Val MAE: 24.493784 | Test MAE: 24.556377
  ⟨R²⟩ (Ang²)     | Val MAE: 54.134514 | Test MAE: 54.251438
  ZPVE (eV)       | Val MAE: 8.633186 | Test MAE: 8.382577
  U₀ (eV)         | Val MAE: 17713.222656 | Test MAE: 17310.322266
  U (eV)          | Val MAE: 17632.406250 | Test MAE: 17232.957031
  H (eV)          | Val MAE: 17768.160156 | Test MAE: 17363.404297
  G (eV)          | Val MAE: 17528.521484 | Test MAE: 17132.367188
  c_v             | Val MAE: 2.522058 | Test MAE: 2.477818
  U₀_atom         | Val MAE: 3.878299 | Test MAE: 3.747461
  U_atom          | Val MAE: 105.391487 | Test MAE: 101.686897
  H_atom          | Val MAE: 104.743935 | Test MAE: 101.012329
  G_atom          | Val MAE: 98.778511 |

...

Train loss (MSE): 0.505478
  μ (D)           | Val MAE: 0.839569 | Test MAE: 0.851215
  α (Ang³)        | Val MAE: 0.531405 | Test MAE: 0.523779
  ε_HOMO (eV)     | Val MAE: 10.014853 | Test MAE: 9.954527
  ε_LUMO (eV)     | Val MAE: 17.155052 | Test MAE: 17.372047
  Δε (eV)         | Val MAE: 20.312193 | Test MAE: 20.233049
  ⟨R²⟩ (Ang²)     | Val MAE: 48.428036 | Test MAE: 48.211937
  ZPVE (eV)       | Val MAE: 5.821439 | Test MAE: 5.603852
  U₀ (eV)         | Val MAE: 11922.255859 | Test MAE: 11483.635742
  U (eV)          | Val MAE: 11859.278320 | Test MAE: 11420.609375
  H (eV)          | Val MAE: 11900.108398 | Test MAE: 11461.893555
  G (eV)          | Val MAE: 11893.817383 | Test MAE: 11453.622070
  c_v             | Val MAE: 1.950373 | Test MAE: 1.899236
  U₀_atom         | Val MAE: 3.233164 | Test MAE: 3.111857
  U_atom          | Val MAE: 88.437172 | Test MAE: 85.131248
  H_atom          | Val MAE: 88.662407 | Test MAE: 85.319618
  G_atom          | Val MAE: 81.393974 | Test

Train loss (MSE): 0.505285
  μ (D)           | Val MAE: 0.839569 | Test MAE: 0.851215
  α (Ang³)        | Val MAE: 0.531405 | Test MAE: 0.523779
  ε_HOMO (eV)     | Val MAE: 10.014853 | Test MAE: 9.954527
  ε_LUMO (eV)     | Val MAE: 17.155052 | Test MAE: 17.372047
  Δε (eV)         | Val MAE: 20.312191 | Test MAE: 20.233049
  ⟨R²⟩ (Ang²)     | Val MAE: 48.428036 | Test MAE: 48.211937
  ZPVE (eV)       | Val MAE: 5.821439 | Test MAE: 5.603852
  U₀ (eV)         | Val MAE: 11922.256836 | Test MAE: 11483.636719
  U (eV)          | Val MAE: 11859.277344 | Test MAE: 11420.608398
  H (eV)          | Val MAE: 11900.108398 | Test MAE: 11461.893555
  G (eV)          | Val MAE: 11893.817383 | Test MAE: 11453.622070
  c_v             | Val MAE: 1.950373 | Test MAE: 1.899236
  U₀_atom         | Val MAE: 3.233164 | Test MAE: 3.111857
  U_atom          | Val MAE: 88.437172 | Test MAE: 85.131248
  H_atom          | Val MAE: 88.662407 | Test MAE: 85.319618
  G_atom          | Val MAE: 81.393974 | Test

Train loss (MSE): 0.505050
  μ (D)           | Val MAE: 0.839569 | Test MAE: 0.851215
  α (Ang³)        | Val MAE: 0.531405 | Test MAE: 0.523779
  ε_HOMO (eV)     | Val MAE: 10.014853 | Test MAE: 9.954527
  ε_LUMO (eV)     | Val MAE: 17.155052 | Test MAE: 17.372047
  Δε (eV)         | Val MAE: 20.312193 | Test MAE: 20.233049
  ⟨R²⟩ (Ang²)     | Val MAE: 48.428036 | Test MAE: 48.211937
  ZPVE (eV)       | Val MAE: 5.821439 | Test MAE: 5.603852
  U₀ (eV)         | Val MAE: 11922.256836 | Test MAE: 11483.636719
  U (eV)          | Val MAE: 11859.278320 | Test MAE: 11420.608398
  H (eV)          | Val MAE: 11900.108398 | Test MAE: 11461.893555
  G (eV)          | Val MAE: 11893.817383 | Test MAE: 11453.623047
  c_v             | Val MAE: 1.950373 | Test MAE: 1.899236
  U₀_atom         | Val MAE: 3.233164 | Test MAE: 3.111857
  U_atom          | Val MAE: 88.437172 | Test MAE: 85.131248
  H_atom          | Val MAE: 88.662407 | Test MAE: 85.319618
  G_atom          | Val MAE: 81.393974 | Test

Train loss (MSE): 0.505622
  μ (D)           | Val MAE: 0.839569 | Test MAE: 0.851215
  α (Ang³)        | Val MAE: 0.531405 | Test MAE: 0.523779
  ε_HOMO (eV)     | Val MAE: 10.014853 | Test MAE: 9.954527
  ε_LUMO (eV)     | Val MAE: 17.155052 | Test MAE: 17.372047
  Δε (eV)         | Val MAE: 20.312193 | Test MAE: 20.233049
  ⟨R²⟩ (Ang²)     | Val MAE: 48.428036 | Test MAE: 48.211937
  ZPVE (eV)       | Val MAE: 5.821439 | Test MAE: 5.603852
  U₀ (eV)         | Val MAE: 11922.255859 | Test MAE: 11483.636719
  U (eV)          | Val MAE: 11859.277344 | Test MAE: 11420.609375
  H (eV)          | Val MAE: 11900.108398 | Test MAE: 11461.893555
  G (eV)          | Val MAE: 11893.817383 | Test MAE: 11453.622070
  c_v             | Val MAE: 1.950373 | Test MAE: 1.899236
  U₀_atom         | Val MAE: 3.233164 | Test MAE: 3.111857
  U_atom          | Val MAE: 88.437172 | Test MAE: 85.131248
  H_atom          | Val MAE: 88.662407 | Test MAE: 85.319618
  G_atom          | Val MAE: 81.393974 | Test

Train loss (MSE): 0.505271
  μ (D)           | Val MAE: 0.839569 | Test MAE: 0.851215
  α (Ang³)        | Val MAE: 0.531405 | Test MAE: 0.523779
  ε_HOMO (eV)     | Val MAE: 10.014853 | Test MAE: 9.954527
  ε_LUMO (eV)     | Val MAE: 17.155052 | Test MAE: 17.372047
  Δε (eV)         | Val MAE: 20.312193 | Test MAE: 20.233049
  ⟨R²⟩ (Ang²)     | Val MAE: 48.428036 | Test MAE: 48.211937
  ZPVE (eV)       | Val MAE: 5.821439 | Test MAE: 5.603851
  U₀ (eV)         | Val MAE: 11922.257812 | Test MAE: 11483.636719
  U (eV)          | Val MAE: 11859.280273 | Test MAE: 11420.610352
  H (eV)          | Val MAE: 11900.110352 | Test MAE: 11461.894531
  G (eV)          | Val MAE: 11893.818359 | Test MAE: 11453.624023
  c_v             | Val MAE: 1.950373 | Test MAE: 1.899236
  U₀_atom         | Val MAE: 3.233164 | Test MAE: 3.111857
  U_atom          | Val MAE: 88.437172 | Test MAE: 85.131241
  H_atom          | Val MAE: 88.662407 | Test MAE: 85.319611
  G_atom          | Val MAE: 81.393967 | Test